In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit_metal as metal
from qiskit_metal import Dict, designs, MetalGUI
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
import numpy as np
from collections import OrderedDict
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.framed_path import RouteFramed
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital

In [3]:
design = designs.DesignPlanar(overwrite_enabled=True)

In [4]:
gui = MetalGUI(design)
design.variables['cpw_width']='10um'
design.variables['cpw_gap']='6um'

# Calculos das capacitancias

In [8]:
l = 200
d = np.linspace(0, 1000, 100)

a0 = 1.62129e-7
a1 = 0.100845
b0 = 0.0592123
b1 = -2.29999e-6
m0 = 0.000600864
m1 = -2.1275e-6
c0 = -0.201206
c1 = 0.00156082
A = a0 + a1*l
B = b0 + b1*l
M = m0 + m1*l
C = c0 + c1*l

C = A*np.exp(-B*d) + M*d + C

plt.plot(d, C)

In [88]:
def Cap(d, l):
    """d (um) - distancia do gap entre os dois metais
        l (um) - comprimento dos metais que ficam paralelos"""
    a0 = 1.62129e-7
    a1 = 0.100845
    b0 = 0.0592123
    b1 = -2.29999e-6
    m0 = 0.000600864
    m1 = -2.1275e-6
    c0 = -0.201206
    c1 = 0.00156082
    A = a0 + a1*l
    B = b0 + b1*l
    M = m0 + m1*l
    C = c0 + c1*l
    
    C = A*np.exp(-B*d) + M*d + C
    return C

Cap(20, 350)

11.317167759621826

# Definindo funções

Primeiro teste

In [64]:
def ressonador_cima(resonator_number, connection1, connection2):
    """
    resonator_number - numero do ressonador, para nomerar o ressonador
    connection = [pos_x, pos_y] em milimetros"""

    OpenToGround(design, f'OTG{resonator_number}_1', options=Dict(
        pos_x=f'{connection1[0]}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    OpenToGround(design, f'OTG{resonator_number}_2', options=Dict(
        pos_x=f'{connection2[0]}mm',
        pos_y=f'{connection2[1]}mm',
        orientation='90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    return RouteFramed(design, f'RES{resonator_number}', options=Dict(
        pin_inputs = Dict(
            start_pin = Dict(
                component = f'OTG{resonator_number}_1',
                pin = 'open'
            ),
            end_pin = Dict(
                component = f'OTG{resonator_number}_2',
                pin = 'open'
            )
        ),
        #total_length='10mm',
        lead=Dict(start_straight='4.15mm', end_straight='4.15mm'),
        fillet='300um',
        hfss_wire_bonds = True
    ))

In [65]:
def ressonador_baixo(resonator_number, connection1, connection2):
    """
    resonator_number - numero do ressonador, para nomerar o ressonador
    connection = [pos_x, pos_y] em milimetros"""

    OpenToGround(design, f'OTG{resonator_number}_1', options=Dict(
        pos_x=f'{connection1[0]}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='-90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    OpenToGround(design, f'OTG{resonator_number}_2', options=Dict(
        pos_x=f'{connection2[0]}mm',
        pos_y=f'{connection2[1]}mm',
        orientation='-90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    return RouteFramed(design, f'RES{resonator_number}', options=Dict(
        pin_inputs = Dict(
            start_pin = Dict(
                component = f'OTG{resonator_number}_1',
                pin = 'open'
            ),
            end_pin = Dict(
                component = f'OTG{resonator_number}_2',
                pin = 'open'
            )
        ),
        total_length='10mm',
        lead=Dict(start_straight='4.15mm', end_straight='4.15mm'),
        fillet='300um',
        hfss_wire_bonds = True
    ))

In [66]:
x = 2.8 # distancia dos ressonadores na horizontal
y = 5.8 # distancia dos ressonadores na vertical
res1 = ressonador_cima(1, [0, 0], [2, 0])
res2 = ressonador_cima(2, [1*x, 0], [1*x+2, 0])
res3 = ressonador_cima(3, [2*x, 0], [2*x+2, 0])
res4 = ressonador_cima(4, [3*x, 0], [3*x+2, 0])
res5 = ressonador_cima(5, [4*x, 0], [4*x+2, 0])
res6 = ressonador_cima(6, [5*x, 0], [5*x+2, 0])
res7 = ressonador_cima(7, [6*x, 0], [6*x+2, 0])
res8 = ressonador_cima(8, [7*x, 0], [7*x+2, 0])

res9 = ressonador_baixo(9, [1.4, -3], [1.4+2, -3])
res10 = ressonador_baixo(10, [1.4+1*x, -3], [1.4+1*x+2, -3])
res11 = ressonador_baixo(11, [1.4+2*x, -3], [1.4+2*x+2, -3])
res12 = ressonador_baixo(12, [1.4+3*x, -3], [1.4+3*x+2, -3])
res13 = ressonador_baixo(13, [1.4+4*x, -3], [1.4+4*x+2, -3])
res14 = ressonador_baixo(14, [1.4+5*x, -3], [1.4+5*x+2, -3])
res15 = ressonador_baixo(15, [1.4+6*x, -3], [1.4+6*x+2, -3])

res16 = ressonador_cima(16, [1.4+0*x, y], [1.4+0*x+2, y])
res17 = ressonador_cima(17, [1.4+1*x, y], [1.4+1*x+2, y])
res18 = ressonador_cima(18, [1.4+2*x, y], [1.4+2*x+2, y])
res19 = ressonador_cima(19, [1.4+3*x, y], [1.4+3*x+2, y])
res20 = ressonador_cima(20, [1.4+4*x, y], [1.4+4*x+2, y])
res21 = ressonador_cima(21, [1.4+5*x, y], [1.4+5*x+2, y])
res22 = ressonador_cima(22, [1.4+6*x, y], [1.4+6*x+2, y])

res23 = ressonador_baixo(23, [0, -3+y], [2, -3+y])
res24 = ressonador_baixo(24, [1*x, -3+y], [1*x+2, -3+y])
res25 = ressonador_baixo(25, [2*x, -3+y], [2*x+2, -3+y])
res26 = ressonador_baixo(26, [3*x, -3+y], [3*x+2, -3+y])
res27 = ressonador_baixo(27, [4*x, -3+y], [4*x+2, -3+y])
res28 = ressonador_baixo(28, [5*x, -3+y], [5*x+2, -3+y])
res29 = ressonador_baixo(29, [6*x, -3+y], [6*x+2, -3+y])
res30 = ressonador_baixo(30, [7*x, -3+y], [7*x+2, -3+y])

res31 = ressonador_cima(31, [0*x, 2*y], [0*x+2, 2*y])
res32 = ressonador_cima(32, [1*x, 2*y], [1*x+2, 2*y])
res33 = ressonador_cima(33, [2*x, 2*y], [2*x+2, 2*y])
res34 = ressonador_cima(34, [3*x, 2*y], [3*x+2, 2*y])
res35 = ressonador_cima(35, [4*x, 2*y], [4*x+2, 2*y])
res36 = ressonador_cima(36, [5*x, 2*y], [5*x+2, 2*y])
res37 = ressonador_cima(37, [6*x, 2*y], [6*x+2, 2*y])
res38 = ressonador_cima(38, [7*x, 2*y], [7*x+2, 2*y])


gui.rebuild()
gui.autoscale()

In [70]:
design.delete_all_components()

# Definindo novas funções

Deixando os ressonadores mais compactos
Espaço ocupado pelo ressonador: 1.2mm em x por 1.3mm em y

In [29]:
def ressonador_cima_novo(resonator_number, connection1):
    """
    resonator_number - numero do ressonador, para nomerar o ressonador
    connection = [pos_x, pos_y] em milimetros"""

    OpenToGround(design, f'OTG{resonator_number}_1', options=Dict(
        pos_x=f'{connection1[0]}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='180',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    OpenToGround(design, f'OTG{resonator_number}_2', options=Dict(
        pos_x=f'{connection1[0] + 1.2}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='0',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    jogsS = OrderedDict()
    jogsS[0] = ["R", '0.2mm']
    jogsS[1] = ["R", '0.35mm']
    jogsS[2] = ["L", '0.2mm']
    jogsS[3] = ["L", '1mm']
    jogsS[4] = ["R", '0.2mm']
    jogsS[5] = ["R", '1mm']
    jogsS[6] = ["L", '0.58mm']
    jogsS[7] = ["L", '0.425mm']
    jogsS[8] = ["R", '0.12mm']

    jogsE = OrderedDict()
    jogsE[0] = ["L", '0.2mm']
    jogsE[1] = ["L", '0.35mm']
    jogsE[2] = ["R", '0.6mm']
    jogsE[3] = ["R", '1mm']
    jogsE[4] = ["L", '0.2mm']
    jogsE[5] = ["L", '1mm']
    jogsE[6] = ["R", '0.18mm']
    jogsE[7] = ["R", '0.425mm']
    jogsE[8] = ["L", '0.12mm']

    return RouteFramed(design, f'RES{resonator_number}', options=Dict(
        pin_inputs = Dict(
            start_pin = Dict(
                component = f'OTG{resonator_number}_1',
                pin = 'open'
            ),
            end_pin = Dict(
                component = f'OTG{resonator_number}_2',
                pin = 'open'
            )
        ),
        #total_length='10mm',
        lead=Dict(start_straight = '0.35mm', end_straight = '0.35mm', start_jogged_extension = jogsS, end_jogged_extension = jogsE),
        fillet='50um',
        hfss_wire_bonds = True
    ))

In [30]:
def ressonador_baixo_novo(resonator_number, connection1):
    """
    resonator_number - numero do ressonador, para nomerar o ressonador
    connection = [pos_x, pos_y] em milimetros"""

    OpenToGround(design, f'OTG{resonator_number}_1', options=Dict(
        pos_x=f'{connection1[0]}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='180',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    OpenToGround(design, f'OTG{resonator_number}_2', options=Dict(
        pos_x=f'{connection1[0] + 1.2}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='0',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    jogsS = OrderedDict()
    jogsS[0] = ["L", '0.2mm']
    jogsS[1] = ["L", '0.35mm']
    jogsS[2] = ["R", '0.2mm']
    jogsS[3] = ["R", '1mm']
    jogsS[4] = ["L", '0.2mm']
    jogsS[5] = ["L", '1mm']
    jogsS[6] = ["R", '0.58mm']
    jogsS[7] = ["R", '0.425mm']
    jogsS[8] = ["L", '0.12mm']

    jogsE = OrderedDict()
    jogsE[0] = ["R", '0.2mm']
    jogsE[1] = ["R", '0.35mm']
    jogsE[2] = ["L", '0.6mm']
    jogsE[3] = ["L", '1mm']
    jogsE[4] = ["R", '0.2mm']
    jogsE[5] = ["R", '1mm']
    jogsE[6] = ["L", '0.18mm']
    jogsE[7] = ["L", '0.425mm']
    jogsE[8] = ["R", '0.12mm']

    return RouteFramed(design, f'RES{resonator_number}', options=Dict(
        pin_inputs = Dict(
            start_pin = Dict(
                component = f'OTG{resonator_number}_1',
                pin = 'open'
            ),
            end_pin = Dict(
                component = f'OTG{resonator_number}_2',
                pin = 'open'
            )
        ),
        #total_length='10mm',
        lead=Dict(start_straight = '0.35mm', end_straight = '0.35mm', start_jogged_extension = jogsS, end_jogged_extension = jogsE),
        fillet='50um',
        hfss_wire_bonds = True
    ))

In [83]:
x = 1.7 # distancia dos ressonadores no eixo x
y = 0.06 # distancia dos ressonadores no eixo y

for i in range(1,13):
    res = ressonador_cima_novo(i, [(i)*x, 0])

# for i in range(14,27):
#      res = ressonador_baixo_novo(i, [0.85+(i-14)*x, y])

# for i in range(27,40):
#     res = ressonador_cima_novo(i, [0.85+(i-27)*x, 0.06+1.3+0.06+1.3])

# for i in range(40,52):
#     res = ressonador_baixo_novo(i, [(i-39)*x, 0.06+1.3+0.06+1.3+0.06])

# for i in range(52,64):
#     res = ressonador_cima_novo(i, [(i-51)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3])

# for i in range(64,77):
#     res = ressonador_baixo_novo(i, [0.85+(i-64)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06])

# for i in range(77,90):
#     res = ressonador_cima_novo(i, [0.85+(i-77)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3])

# for i in range(90,102):
#     res = ressonador_baixo_novo(i, [(i-89)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06])

# for i in range(102,114):
#     res = ressonador_cima_novo(i, [(i-101)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3])

# for i in range(114,127):
#     res = ressonador_baixo_novo(i, [0.85+(i-114)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06])

# for i in range(127,140):
#     res = ressonador_cima_novo(i, [0.85+(i-127)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3])

# for i in range(140, 152):
#     res = ressonador_baixo_novo(i, [(i-139)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06])

# for i in range(152,164):
#     res = ressonador_cima_novo(i, [(i-151)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3])

# for i in range(164,177):
#     res = ressonador_baixo_novo(i, [0.85+(i-164)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06])

# for i in range(177,190):
#     res = ressonador_cima_novo(i, [0.85+(i-177)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3])

for i in range(190,202):
    res = ressonador_baixo_novo(i, [(i-189)*x, 0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06+1.3+0.06])

# for i in range(102,114):
#     res = ressonador_cima_novo(i, [(i-)])

gui.rebuild()
gui.autoscale()

In [76]:
# linha 1
pad1 = LaunchpadWirebond(design, 'PAD1', options=Dict(
    pos_x='0.4mm',
    pos_y='20.2mm',
    orientation='-90',
    lead_length='25um',
    pad_width='350um',
    pad_height='350um',
    pad_gap='150um',
    taper_height='370um'
))

pad2 = LaunchpadWirebond(design, 'PAD2', options=Dict(
    pos_x='22.9mm',
    pos_y='20.2mm',
    orientation='-90',
    lead_length='25um',
    pad_width='350um',
    pad_height='350um',
    pad_gap='150um',
    taper_height='370um'
))

pad3 = LaunchpadWirebond(design, 'PAD3', options=Dict(
    pos_x='0.45mm',
    pos_y='-1.5mm',
    orientation='0',
    lead_length='25um',
    pad_width='350um',
    pad_height='350um',
    pad_gap='150um',
    taper_height='370um'
))

pad4 = LaunchpadWirebond(design, 'PAD4', options=Dict(
    pos_x='22.85mm',
    pos_y='-1.5mm',
    orientation='180',
    lead_length='25um',
    pad_width='350um',
    pad_height='350um',
    pad_gap='150um',
    taper_height='370um'
))

gui.rebuild()
gui.autoscale()

In [120]:
design.delete_all_components()

## Hexagono

In [31]:
res = ressonador_cima_novo(1, [0,0])
res = ressonador_baixo_novo(2, [-0.85, 0.06])
res = ressonador_baixo_novo(3, [0.85, 0.06])
res = ressonador_cima_novo(4, [-0.85, 0.06+1.3+0.06+1.3])
res = ressonador_cima_novo(5, [0.85, 0.06+1.3+0.06+1.3])
res = ressonador_baixo_novo(6, [0,0.06+1.3+0.06+1.3+0.06])

gui.rebuild()
gui.autoscale()

# Versão 3 dos ressonadores

In [92]:
def ressonador_cima_v3(resonator_number, connection1):
    """
    resonator_number - numero do ressonador, para nomerar o ressonador
    connection = [pos_x, pos_y] em milimetros"""

    OpenToGround(design, f'OTG{resonator_number}_1', options=Dict(
        pos_x=f'{connection1[0]+0.132}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='180',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    OpenToGround(design, f'OTG{resonator_number}_2', options=Dict(
        pos_x=f'{connection1[0] + 1.596}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='0',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    jogsS = OrderedDict()
    jogsS[0] = ["R", '0.15mm']
    jogsS[1] = ["R", '0.482mm']
    jogsS[2] = ["L", '0.15mm']
    jogsS[3] = ["L", '1.578mm']
    jogsS[4] = ["R", '0.15mm']
    jogsS[5] = ["R", '1.578mm']
    jogsS[6] = ["L", '0.45mm']
    jogsS[7] = ["L", '0.587mm']
    jogsS[8] = ["R30", '0.13mm']

    jogsE = OrderedDict()
    jogsE[0] = ["L", '0.15mm']
    jogsE[1] = ["L", '0.482mm']
    jogsE[2] = ["R", '0.45mm']
    jogsE[3] = ["R", '1.578mm']
    jogsE[4] = ["L", '0.15mm']
    jogsE[5] = ["L", '1.578mm']
    jogsE[6] = ["R", '0.15mm']
    jogsE[7] = ["R", '0.587mm']
    jogsE[8] = ["L30", '0.13mm']

    return RouteFramed(design, f'RES{resonator_number}', options=Dict(
        pin_inputs = Dict(
            start_pin = Dict(
                component = f'OTG{resonator_number}_1',
                pin = 'open'
            ),
            end_pin = Dict(
                component = f'OTG{resonator_number}_2',
                pin = 'open'
            )
        ),
        #total_length='10mm',
        lead=Dict(start_straight = '0.35mm', end_straight = '0.35mm', start_jogged_extension = jogsS, end_jogged_extension = jogsE),
        fillet='50um',
        hfss_wire_bonds = True
    ))

In [93]:
def ressonador_baixo_v3(resonator_number, connection1):
    """
    resonator_number - numero do ressonador, para nomerar o ressonador
    connection = [pos_x, pos_y] em milimetros"""

    OpenToGround(design, f'OTG{resonator_number}_1', options=Dict(
        pos_x=f'{connection1[0]+0.132}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='180',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    OpenToGround(design, f'OTG{resonator_number}_2', options=Dict(
        pos_x=f'{connection1[0] + 1.596}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='0',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    jogsS = OrderedDict()
    jogsS[0] = ["L", '0.15mm']
    jogsS[1] = ["L", '0.482mm']
    jogsS[2] = ["R", '0.15mm']
    jogsS[3] = ["R", '1.578mm']
    jogsS[4] = ["L", '0.15mm']
    jogsS[5] = ["L", '1.578mm']
    jogsS[6] = ["R", '0.45mm']
    jogsS[7] = ["R", '0.587mm']
    jogsS[8] = ["L30", '0.13mm']

    jogsE = OrderedDict()
    jogsE[0] = ["R", '0.15mm']
    jogsE[1] = ["R", '0.482mm']
    jogsE[2] = ["L", '0.45mm']
    jogsE[3] = ["L", '1.578mm']
    jogsE[4] = ["R", '0.15mm']
    jogsE[5] = ["R", '1.578mm']
    jogsE[6] = ["L", '0.15mm']
    jogsE[7] = ["L", '0.587mm']
    jogsE[8] = ["R30", '0.13mm']

    return RouteFramed(design, f'RES{resonator_number}', options=Dict(
        pin_inputs = Dict(
            start_pin = Dict(
                component = f'OTG{resonator_number}_1',
                pin = 'open'
            ),
            end_pin = Dict(
                component = f'OTG{resonator_number}_2',
                pin = 'open'
            )
        ),
        #total_length='10mm',
        lead=Dict(start_straight = '0.35mm', end_straight = '0.35mm', start_jogged_extension = jogsS, end_jogged_extension = jogsE),
        fillet='50um',
        hfss_wire_bonds = True
    ))

## 1 ressonador

In [137]:
res = ressonador_cima_v3(1, [0,0])

pad1 = LaunchpadWirebond(design, 'PAD1', options=Dict(
    pos_x='-0.68mm',
    pos_y='-0.2mm',
    orientation='0',
    lead_length='25um',
    pad_width='350um',
    pad_height='350um',
    pad_gap='150um',
    taper_height='370um'
))

otg1 = OpenToGround(design, 'OTG1', options=Dict(
    pos_x='0.482mm',
    pos_y='0.04mm',
    orientation='0',
    width='10um',
    gap='6um',
    termination_gap='6um'
))

line1 = RouteStraight(design, 'LINE1', options=Dict(
    pin_inputs = Dict(
        start_pin = Dict(
            component = 'OTG1',
            pin = 'open'
        ),
        end_pin = Dict(
            component = 'PAD1',
            pin = 'tie'
        )
    ),
    lead = Dict(start_straight=0.5, end_straight=0.15),
    fillet='150um'
))

pad2 = LaunchpadWirebond(design, 'PAD2', options=Dict(
    pos_x='2.4mm',
    pos_y='-0.2mm',
    orientation='180',
    lead_length='25um',
    pad_width='350um',
    pad_height='350um',
    pad_gap='150um',
    taper_height='370um'
))

otg2 = OpenToGround(design, 'OTG2', options=Dict(
    pos_x='1.246mm',
    pos_y='0.04mm',
    orientation='180',
    width='10um',
    gap='6um',
    termination_gap='6um'
))

line2 = RouteStraight(design, 'LINE2', options=Dict(
    pin_inputs = Dict(
        start_pin = Dict(
            component = 'OTG2',
            pin = 'open'
        ),
        end_pin = Dict(
            component = 'PAD2',
            pin = 'tie'
        )
    ),
    lead = Dict(start_straight=0.5, end_straight=0.15),
    fillet='150um'
))

gui.rebuild()
gui.autoscale()

## Hexagono

In [117]:
res = ressonador_cima_v3(1, [0,0])
res = ressonador_baixo_v3(2, [-1.114, 0.04])
res = ressonador_baixo_v3(3, [1.114, 0.04])
res = ressonador_cima_v3(4, [-1.114, 0.04+0.965+0.04+0.965])
res = ressonador_cima_v3(5, [1.114, 0.04+0.965+0.04+0.965])
res = ressonador_baixo_v3(6, [0,0.04+0.965+0.04+0.965+0.04])

pad1 = LaunchpadWirebond(design, 'PAD1', options=Dict(
    pos_x='0.85mm',
    pos_y='-3.025mm',
    orientation='90',
    lead_length='25um',
    pad_width='350um',
    pad_height='350um',
    pad_gap='150um',
    taper_height='370um'
))

otg1 = OpenToGround(design, 'OTG1', options=Dict(
    pos_x='0.7mm',
    pos_y='-1.005mm',
    orientation='180',
    width='10um',
    gap='6um',
    termination_gap='6um'
))

line1 = RouteStraight(design, 'LINE1', options=Dict(
    pin_inputs = Dict(
        start_pin = Dict(
            component = 'OTG1',
            pin = 'open'
        ),
        end_pin = Dict(
            component = 'PAD1',
            pin = 'tie'
        )
    ),
    lead = Dict(start_straight=0.6, end_straight=0.5),
    fillet='200um'
))

pad2 = LaunchpadWirebond(design, 'PAD2', options=Dict(
    pos_x='0.85mm',
    pos_y='5.055mm',
    orientation='-90',
    lead_length='25um',
    pad_width='350um',
    pad_height='350um',
    pad_gap='150um',
    taper_height='370um'
))

otg2 = OpenToGround(design, 'OTG2', options=Dict(
    pos_x='0.7mm',
    pos_y='3.055mm',
    orientation='180',
    width='10um',
    gap='6um',
    termination_gap='6um'
))

line2 = RouteStraight(design, 'LINE2', options=Dict(
    pin_inputs = Dict(
        start_pin = Dict(
            component = 'OTG2',
            pin = 'open'
        ),
        end_pin = Dict(
            component = 'PAD2',
            pin = 'tie'
        )
    ),
    lead = Dict(start_straight=0.6, end_straight=0.5),
    fillet='200um'
))

gui.rebuild()
gui.autoscale()

# Render to GDS

In [138]:
design.chips.main.size['size_x'] = '5mm'
design.chips.main.size['size_y'] = '5mm'
design.chips.main.size['center_x'] = '0.87mm'
design.chips.main.size['center_y'] = '-0.2mm'
design.chips.main

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.87mm',
  'center_y': '-0.2mm',
  'center_z': '0.0mm',
  'size_x': '5mm',
  'size_y': '5mm',
  'size_z': '-750um',
  'sample_holder_top': '890um',
  'sample_holder_bottom': '1650um'}}

In [139]:
design.renderers.gds.export_to_gds('Teste Hexagon lattice.gds')

07:36PM 44s WARNING [_import_junctions_to_one_cell]: Not able to find file:"../resources/Fake_Junctions.GDS".  Not used to replace junction. Checked directory:"c:\Users\edhan\Desktop\Git hub\Projeto-Valentina\resources".


1